# 05 – Conclude & Evaluate

**Projekt:** WealthScope AI 1.0  
**Methode:** QUA³CK · reproduzierbarer Out-of-Time-Benchmark  
**Hinweis:** Wissenschaftlicher Prototyp, keine Anlageberatung.

## Ergebnislogik

Accuracy allein ist wegen der Mehrheitsklasse irreführend. Deshalb werden
Balanced Accuracy, ROC-AUC, Average Precision, Konfusionsmatrix und
Walk-forward-Stabilität gemeinsam bewertet.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "wealthscope_features.parquet"
DIAGNOSTICS_PATH = PROJECT_ROOT / "models" / "diagnostics.json"
EXPERIMENTS_PATH = PROJECT_ROOT / "models" / "validation_experiments.json"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Datensatz fehlt: {DATA_PATH}")

df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
print(f"Daten: {len(df):,} Zeilen × {len(df.columns)} Spalten")
display(df.head(3))

In [ ]:
diagnostics = json.loads(DIAGNOSTICS_PATH.read_text(encoding="utf-8"))
m = diagnostics["test_metrics"]
summary = pd.Series({
    "Random-Forest Accuracy": m["accuracy"],
    "Mehrheitsbaseline Accuracy": m["majority_baseline"],
    "Balanced Accuracy": m["balanced_accuracy"],
    "ROC-AUC": m["roc_auc"],
    "Average Precision": m["average_precision"],
    "Walk-forward ROC-AUC": diagnostics["cross_validation"]["auc_mean"],
})
summary.to_frame("Wert").style.format("{:.3f}")

In [ ]:
cm = np.array(diagnostics["confusion_matrix"]["counts"])
fig, ax = plt.subplots(figsize=(4.5, 3.8))
im = ax.imshow(cm, cmap="Greens")
for (i, j), value in np.ndenumerate(cm):
    ax.text(j, i, f"{value:,}", ha="center", va="center")
ax.set_xticks([0, 1], ["0", "1"])
ax.set_yticks([0, 1], ["0", "1"])
ax.set_xlabel("Vorhersage")
ax.set_ylabel("Wahr")
ax.set_title("Konfusionsmatrix · Out-of-Time-Test")
plt.colorbar(im, ax=ax, fraction=.045)
plt.show()

In [ ]:
hypotheses = pd.DataFrame([
    ["H1", "falsifiziert",
     f"RF AUC {m['roc_auc']:.3f}; Walk-forward {diagnostics['cross_validation']['auc_mean']:.3f}"],
    ["H2", "im Prototyp umgesetzt", "App, Methodik, Export und Lernstudio; Nutzerstudie offen"],
    ["H3", "teilweise plausibel", "Orientierung verbessert; wirtschaftlicher Nutzen nicht bewiesen"],
], columns=["Hypothese", "Urteil", "Begründung"])
hypotheses

H1 ist **falsifiziert, nicht ungeprüft**. Die Hypothese war so formuliert, dass
sie scheitern konnte – genau das macht sie zu einer wissenschaftlichen Aussage.
Ein Negativergebnis ist allerdings nur dann ein Befund, wenn die naheliegenden
Gegenerklärungen ausgeschlossen sind. Genau das prüft die nächste Zelle.

In [ ]:
experiments = json.loads(EXPERIMENTS_PATH.read_text(encoding="utf-8"))
cap = experiments["capacity_sweep_summary"]
split = experiments["split_comparison_summary"]
lc = json.loads((PROJECT_ROOT / "models" / "learning_curve.json").read_text(encoding="utf-8"))
val_trend = lc["val_scores_mean"][-1] - lc["val_scores_mean"][0]
data_factor = lc["train_sizes_abs"][-1] / lc["train_sizes_abs"][0]

checks = pd.DataFrame([
    ["Zu wenig Modellkapazität?", "ausgeschlossen",
     f"Test-AUC-Spanne über 7 Stufen nur {cap['test_roc_auc_span']:.4f} "
     f"({cap['test_roc_auc_min']:.4f}–{cap['test_roc_auc_max']:.4f})"],
    ["Zu wenig Daten?", "ausgeschlossen",
     f"{data_factor:.1f}-fache Trainingsmenge verändert die Validierung um "
     f"{val_trend:+.4f}"],
    ["Ist die Aufteilung der Hebel?", "ja – und zwar der einzige",
     f"naiver Zufalls-Split: AUC {split['leaky_roc_auc']:.4f} statt "
     f"{split['reference_roc_auc']:.4f} → scheinbares Signal "
     f"{split['apparent_signal_factor']:.1f}x größer"],
], columns=["Gegenerklärung", "Ergebnis", "Belegte Messung"])
checks.style.hide(axis="index")

## Fazit

Die Out-of-Time-AUC des Random Forest beträgt rund **0,519** – kein belastbares
Handelssignal. Das eigentliche Ergebnis dieses Projekts ist deshalb nicht das
Modell, sondern eine **Messung**: Wie viel verwertbare Information tragen rein
kursbasierte technische Indikatoren? Antwort: nahezu keine, belegt an 190.527
Beobachtungen mit elf Jahren unangetastetem Testzeitraum.

Zwei Gegenerklärungen wurden ausgeschlossen (Kapazität, Datenmenge), und der
tatsächliche Hebel wurde beziffert: Ein naiver Zufalls-Split hätte dieselben
Daten mit AUC 0,581 bewertet – ein rund **4,2-mal größeres scheinbares Signal**.
Damit ist die niedrige Kennzahl kein Qualitätsmangel, sondern der Nachweis, dass
dieser Fehler nicht gemacht wurde. Das ist eine nachgerechnete, nicht bloß
zitierte Bestätigung der Effizienzmarkthypothese (Fama 1970).

### Nächste Schritte

- echter Forward-Test mit neueren Daten;
- Transaktionskosten, Slippage und Steuern;
- Makro-, Fundamental- und Sentimentvariablen (der einzige Hebel, der die
  Validierungskurve heben könnte);
- formale Nutzerstudie zur Verständlichkeit.